# Force and Motion


In [1]:
import sympy as sp
from sympy.physics.mechanics import *
import sympy.physics.vector as spv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import IPython

# Import IPython display for proper LaTeX formatting
from IPython.display import Math

# Initialize symbols
sp.init_printing()

# Enable dot notation printing for dynamicsymbols
from sympy.physics.vector.printing import init_vprinting
init_vprinting(use_latex='mathjax')

In [2]:
def reference_frame(
    frame: str,
    x=r'\imath', y=r'\jmath', z=r'\mathbf k'
) -> ReferenceFrame:
    return ReferenceFrame(
        frame, latexs=(
            fr'\; {{}}^\mathcal {frame} \hat {x}',
            fr'\;{{}}^\mathcal {frame} \hat {y}',
            fr'\: {{}}^\mathcal {frame} \hat {{z}}'
        )
    )


def reference_frame_polar(name):
    return reference_frame(name, x=r"r", y=r"\theta", z=r"e_z")

In [3]:
t = dynamicsymbols._t
N = reference_frame("N")
P = reference_frame_polar("P")
theta = dynamicsymbols("theta")
P.orient_axis(N, theta, N.z)

r = sp.Symbol("r", positive=True)
vec_r = r * P.x
vec_theta = P.y
assert vec_r.dot(vec_theta) == 0

PN_values = {P.x: P.x.express(N), P.y: P.y.express(N)}
display(
    Math(
        r"\boxed{"
        + r" \text{Polar to Cartesian: } \\[5pt] \\"
        + r"\begin{aligned}"
        + sp.latex(P.x)
        + " &= "
        + sp.latex(PN_values[P.x])
        + r" \\ "
        + sp.latex(P.y)
        + " &= "
        + sp.latex(PN_values[P.y])
        + r"\end{aligned}"
        + r"}"
    )
)

# Velocity
vec_v = vec_r.dt(N)
vec_a = vec_v.dt(N)
display(
    Math(
        r" \text{Velocity and Acceleration in Polar Coordinates: } \\[5pt] \\"
        + r"\boxed{"
        + r"\begin{aligned}"
        + r"\vec{v}"
        + " &= "
        + spv.vlatex(vec_v)
        + r" \\ "
        + r"\vec{a}"
        + " &= "
        + spv.vlatex(vec_a)
        + r"\end{aligned}"
        + r"}"
    )
)

# Constaint angular velocity omega, angular acceleration 0
omega = sp.Symbol("omega", positive=True)

values = {theta.diff(t): omega, theta.diff(t, 2): 0}
display(
    Math(
        r"\text{Uniform Circular Motion:} \\[5pt] \\"
        + r"\boxed{"
            + r"\begin{aligned}"
                + r"\vec{v} &= "
                + spv.vlatex(vec_v.subs(values))
                + r" && \text{where } \omega = "+ spv.vlatex(theta.diff(t)) + r" \text{ is constant} \\ "
                + r"\vec{a} &= "
                + spv.vlatex(vec_a.subs(values))
                + r" && \text{where } " + spv.vlatex(theta.diff(t, 2)) + r" = 0" + r" \\ "
            + r"\end{aligned}"
        + r"}"
    ))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Show that for Uniform Circular Motion

$$\boxed{\frac{d \hat{\theta}}{dt} = -\frac{d \theta}{dt}\hat{r}}$$

In [4]:
t = dynamicsymbols._t
N = reference_frame("N")
P = reference_frame_polar("P")
theta = dynamicsymbols("theta")
P.orient_axis(N, theta, N.z)


display(Math(r"\frac{d\hat{\theta}}{dt} =" + spv.vlatex(P.y.diff(t, N).express(P).simplify())))

<IPython.core.display.Math object>

## 11.2 WE - Car on a Banked Turn

A car of mass $m$ is turning on a banked curve of angle $\phi$ with respect to the horizontal. The curve is icy and friction between the tires and the surface is negligible. The curve has a radius $r$. What is the speed $v$ at which the car can turn safely? Express your answer in terms of $m$, $\phi$, and $r$.

In [5]:
m, phi, r, g = sp.symbols("m phi r g", positive=True)
m, phi, r, g

t = dynamicsymbols._t
theta = dynamicsymbols("theta")

N = reference_frame("N")

P = reference_frame_polar("P")
P.orient_axis(N, theta, N.z)

NmB = sp.Symbol("N_mB", positive=True)

vec_FmE = m * g * (-P.y)
vec_NmB = NmB * (sp.cos(sp.pi / 2 + phi) * P.x + sp.cos(phi) * P.y)
vec_F_total = vec_FmE + vec_NmB

vec_r = r * P.x
vec_v = vec_r.diff(t, N)
vec_a = vec_v.diff(t, N).subs(theta.diff(t, 2), 0)

display(
    Math(
        r"\boxed{\begin{aligned}"
        + r"\vec{r} &= "
        + spv.vlatex(vec_r)
        + r"\\"
        + r"\vec{v} &= "
        + spv.vlatex(vec_v)
        + r"\\"
        + r"\vec{a} &= "
        + spv.vlatex(vec_a) + r"\;,&& \text{where } \ddot{\theta} = 0"
        + r"\end{aligned}}"
    )
)

N2Leq = sp.Eq(vec_F_total.to_matrix(N), m * vec_a.to_matrix(N))
result = sp.solve(N2Leq, [NmB, theta.diff(t)], dict=True)

result_vec_v = [
    vec_v.subs(_).subs(theta.diff(t, 2), 0).magnitude().simplify() for _ in result
]
display(
    Math(
        r"\boxed{\begin{aligned}"
        + r"|\vec{v}| &= "
        + spv.vlatex(result_vec_v[0].simplify())
        + r"\end{aligned}}"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Problem Set 3